# Bulgarian phoneme FastSpeech2 inference

Run this notebook in Colab from the same Google Drive directory used by `Bulgarian_Phoneme_A100_Colab.ipynb`.

It uses the trained A100 checkpoints in `output_prosody_v2`, restores the small phoneme assets archive, restores only the minimal preprocessed metadata needed for inference, and calls the repo's shared Bulgarian phonemization path.

In [ ]:
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Use the same values as in the A100 training notebook.
REPO_URL = 'https://github.com/YOUR_USER/FastSpeech2.git'
BRANCH = 'phoneme-mfa'
DRIVE_DIR = '/content/drive/MyDrive/fs2_bg_phone'

# Keep False unless repo_commit_prosody_v2.txt already points to a commit
# that contains this inference notebook and tools/infer_bulgarian.py.
PIN_TO_A100_COMMIT = False

# If None, the notebook uses the latest checkpoint from output_prosody_v2/ckpt/Bulgarian.
RESTORE_STEP = None

TEXT = 'Аз мога да говоря на български език и литература! Вижте колко добре говоря вече.'

# Larger duration_control => slower speech. 1.05-1.25 is usually a sane range for this model.
DURATION_CONTROL = 1.15
PITCH_CONTROL = 1.0
ENERGY_CONTROL = 1.0

# Use our repo-local Bulgarian normalizer before inference.
# contextual expands dates, years, currency, measurements, phones and abbreviations;
# legacy keeps the older num2wordBg-only behavior.
USE_CUSTOM_TEXT_NORMALIZER = True
TEXT_NORMALIZER_MODE = 'contextual'  # contextual | legacy | none
SHOW_NORMALIZATION_PREVIEW = True

# Vocoder switch:
#   'default'   -> repo's universal HiFi-GAN
#   'finetuned' -> your Drive checkpoint below; use g_..., not do_...
VOCODER_MODE = 'default'
FINETUNED_VOCODER_PATH = './hifigan_finetune/g_00135000'

# Optional explicit output basename. If None, each run gets a timestamped name.
OUTPUT_ID = None

# Keep this True if you want unknown words to be phonemized by MFA G2P.
# If all words are already in lexicon/g2p_cache, inference may work without it.
INSTALL_MFA_G2P = True

In [ ]:
import os, shutil, subprocess
from pathlib import Path

os.chdir('/content')
shutil.rmtree('FastSpeech2', ignore_errors=True)
subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, 'FastSpeech2'], check=True)
os.chdir('/content/FastSpeech2')

commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
commit_file = f'{DRIVE_DIR}/repo_commit_prosody_v2.txt'
training_commit = open(commit_file).read().strip() if os.path.isfile(commit_file) else None
if PIN_TO_A100_COMMIT and training_commit:
    expected = training_commit
    if commit != expected:
        subprocess.run(['git', 'checkout', '--detach', expected], check=True)
        commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
    assert commit == expected, f'Inference must use {expected}, got {commit}'
elif not training_commit:
    open(commit_file, 'w').write(commit + '\n')
    training_commit = commit

print('repo commit used for inference:', commit)
print('A100 training commit recorded in Drive:', training_commit)

In [ ]:
import os, shutil, subprocess, sys, zipfile
from pathlib import Path

import torch

packages = ['librosa', 'pyworld', 'scikit-learn', 'matplotlib', 'soundfile', 'PyYAML', 'tqdm']
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *packages], check=True)

VOCODER_TARGET = Path('hifigan/generator_universal.pth.tar')
DEFAULT_VOCODER = Path('hifigan/generator_universal.default.pth.tar')
DEFAULT_VOCODER_ZIP = Path('hifigan/generator_universal.pth.tar.zip')


def extract_default_vocoder():
    assert DEFAULT_VOCODER_ZIP.is_file(), f'Missing {DEFAULT_VOCODER_ZIP}'
    DEFAULT_VOCODER.parent.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(DEFAULT_VOCODER_ZIP) as zf:
        member = next(
            (name for name in zf.namelist() if name.endswith('generator_universal.pth.tar')),
            None,
        )
        assert member is not None, f'{DEFAULT_VOCODER_ZIP} does not contain generator_universal.pth.tar'
        with zf.open(member) as src, open(DEFAULT_VOCODER, 'wb') as dst:
            shutil.copyfileobj(src, dst)


def resolve_drive_path(value):
    path = Path(value).expanduser()
    if path.is_absolute():
        return path
    return Path(DRIVE_DIR) / path


def validate_generator_checkpoint(path):
    ckpt = torch.load(path, map_location='cpu', weights_only=False)
    if not isinstance(ckpt, dict) or 'generator' not in ckpt:
        keys = list(ckpt.keys())[:30] if isinstance(ckpt, dict) else type(ckpt).__name__
        raise ValueError(
            f'{path} is not a HiFi-GAN generator checkpoint. '
            f'Expected key "generator". Got keys/type: {keys}. '
            'For inference use g_00135000, not do_00135000.'
        )
    del ckpt


# Always refresh the default copy so switching back from a fine-tuned vocoder is deterministic.
extract_default_vocoder()

mode = VOCODER_MODE.lower().strip()
if mode == 'default':
    shutil.copy2(DEFAULT_VOCODER, VOCODER_TARGET)
    active_vocoder = DEFAULT_VOCODER
elif mode in {'finetuned', 'custom'}:
    active_vocoder = resolve_drive_path(FINETUNED_VOCODER_PATH)
    assert active_vocoder.is_file(), f'Missing fine-tuned vocoder: {active_vocoder}'
    validate_generator_checkpoint(active_vocoder)
    shutil.copy2(active_vocoder, VOCODER_TARGET)
else:
    raise ValueError("VOCODER_MODE must be 'default' or 'finetuned'")

validate_generator_checkpoint(VOCODER_TARGET)
ACTIVE_VOCODER_PATH = str(active_vocoder)
print('Python dependencies OK')
print('VOCODER_MODE:', VOCODER_MODE)
print('Using vocoder:', active_vocoder)
print('Runtime vocoder file:', VOCODER_TARGET)

In [ ]:
import os, shutil, zipfile
from pathlib import Path

drive_dir = Path(DRIVE_DIR)
assets_zip = drive_dir / 'phoneme_assets.zip'
feature_zip = drive_dir / 'preprocessed_Bulgarian_prosody_v2.zip'
ckpt_dir = drive_dir / 'output_prosody_v2' / 'ckpt' / 'Bulgarian'
result_dir = drive_dir / 'inference_results'
result_dir.mkdir(parents=True, exist_ok=True)

assert assets_zip.is_file(), f'Missing {assets_zip}'
assert feature_zip.is_file(), f'Missing {feature_zip}'
assert ckpt_dir.is_dir(), f'Missing checkpoint directory {ckpt_dir}'

# Runtime lexicon, phone inventory sidecar, punctuation metadata, validation marker.
shutil.unpack_archive(str(assets_zip), '.')

# For inference we do not need all mel/duration/pitch/energy arrays; stats.json is enough
# for pitch/energy denormalization in the plot, and g2p_cache.json is useful if present.
shutil.rmtree('preprocessed_data/Bulgarian', ignore_errors=True)
wanted_suffixes = {
    'Bulgarian/stats.json',
    'Bulgarian/speakers.json',
    'Bulgarian/linguistic_abi.json',
    'Bulgarian/g2p_cache.json',
}
with zipfile.ZipFile(feature_zip) as zf:
    names = zf.namelist()
    extracted = []
    for member in names:
        normalized = member.strip('/')
        if any(normalized.endswith(suffix) for suffix in wanted_suffixes):
            if normalized.startswith('preprocessed_data/'):
                zf.extract(member, '.')
            else:
                zf.extract(member, 'preprocessed_data')
            extracted.append(member)

stats = Path('preprocessed_data/Bulgarian/stats.json')
assert stats.is_file(), 'Could not restore preprocessed_data/Bulgarian/stats.json'

drive_g2p_cache = drive_dir / 'g2p_cache_prosody_v2.json'
local_g2p_cache = Path('preprocessed_data/Bulgarian/g2p_cache.json')
if drive_g2p_cache.is_file():
    local_g2p_cache.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(drive_g2p_cache, local_g2p_cache)
    print('restored Drive G2P cache:', drive_g2p_cache)

checkpoints = sorted(ckpt_dir.glob('*.pth.tar'), key=lambda p: int(p.stem.split('.')[0]) if p.stem.split('.')[0].isdigit() else -1)
assert checkpoints, f'No checkpoints found in {ckpt_dir}'
print('restored inference assets:')
print(' ', stats)
print('latest checkpoint:', checkpoints[-1])
print('result dir:', result_dir)

In [ ]:
import os, subprocess, tempfile
from pathlib import Path

MAMBA_ROOT = '/content/micromamba'
MFA_ROOT_DIR = '/content/mfa_root'
MICROMAMBA = '/content/bin/micromamba'
MFA_CMD = f'{MAMBA_ROOT}/envs/mfa/bin/mfa'
MFA_BIN = str(Path(MFA_CMD).parent)
MFA_SPECS = [
    'python=3.11',
    'montreal-forced-aligner=3.3.9',
    'kalpy>=0.9,<0.10',
    'openfst',
    'pynini',
]


def run_logged(cmd, *, check=True, env=None):
    print('RUN:', ' '.join(map(str, cmd)))
    result = subprocess.run(
        cmd,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        env=env,
    )
    if result.stdout:
        print(result.stdout)
    if check and result.returncode:
        raise subprocess.CalledProcessError(result.returncode, cmd, result.stdout)
    return result


def mfa_env():
    env = os.environ.copy()
    env['PATH'] = MFA_BIN + os.pathsep + env['PATH']
    env['MFA_ROOT_DIR'] = MFA_ROOT_DIR
    env['MAMBA_ROOT_PREFIX'] = MAMBA_ROOT
    return env


def create_mfa_env(*, force=False):
    if force:
        run_logged([MICROMAMBA, 'remove', '-y', '-n', 'mfa', '--all'], check=False)
    run_logged([
        MICROMAMBA, 'create', '-y', '-n', 'mfa', '-c', 'conda-forge',
        '--strict-channel-priority', *MFA_SPECS
    ])


def repair_mfa_env():
    run_logged([
        MICROMAMBA, 'install', '-y', '-n', 'mfa', '-c', 'conda-forge',
        '--strict-channel-priority', *MFA_SPECS
    ])


def mfa_import_smoke(env):
    python = str(Path(MFA_BIN) / 'python')
    if not Path(python).is_file():
        return False
    result = run_logged(
        [
            python,
            '-c',
            'from kalpy.fstext.lexicon import G2PCompiler; import montreal_forced_aligner; print("MFA imports OK")',
        ],
        check=False,
        env=env,
    )
    return result.returncode == 0


def mfa_g2p_smoke(env):
    with tempfile.TemporaryDirectory() as tmp:
        words = Path(tmp) / 'words.txt'
        out = Path(tmp) / 'g2p.dict'
        words.write_text('тест\n', encoding='utf-8')
        result = run_logged(
            [MFA_CMD, 'g2p', str(words), 'bulgarian_mfa', str(out), '--clean', '--sorted'],
            check=False,
            env=env,
        )
        if result.returncode == 0 and out.is_file() and out.read_text(encoding='utf-8').strip():
            print('MFA G2P smoke test OK')
            return True
        print('MFA G2P smoke test FAILED')
        return False


if INSTALL_MFA_G2P:
    os.environ['MAMBA_ROOT_PREFIX'] = MAMBA_ROOT
    os.environ['MFA_ROOT_DIR'] = MFA_ROOT_DIR
    os.environ['PATH'] = MFA_BIN + os.pathsep + os.environ['PATH']

    if not Path(MICROMAMBA).is_file():
        subprocess.run(
            ['bash', '-lc', 'curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xvj -C /content bin/micromamba'],
            check=True,
        )

    if not Path(MFA_CMD).is_file():
        create_mfa_env(force=True)
    env = mfa_env()
    if not Path(MFA_BIN, 'fstcompile').is_file():
        repair_mfa_env()
        env = mfa_env()
    if not mfa_import_smoke(env):
        print('MFA Python imports are broken; recreating the MFA env...')
        create_mfa_env(force=True)
        env = mfa_env()

    assert Path(MFA_BIN, 'fstcompile').is_file(), 'OpenFST fstcompile missing from MFA env'
    assert mfa_import_smoke(env), 'MFA import smoke test failed after env rebuild'

    download = run_logged(
        [MFA_CMD, 'model', 'download', 'g2p', 'bulgarian_mfa'],
        check=False,
        env=env,
    )
    if download.returncode:
        print('MFA model download returned non-zero. Testing whether the model is already usable...')

    if not mfa_g2p_smoke(env):
        raise RuntimeError(
            'MFA G2P model is not usable. If download failed because of Colab/network, '
            'rerun this cell. If you do not need OOV words right now, set INSTALL_MFA_G2P = False.'
        )

    print('MFA G2P OK:', MFA_CMD)
else:
    MFA_CMD = 'mfa'
    MFA_BIN = ''
    print('MFA G2P install skipped; inference works only when every word is in lexicon/cache.')

In [ ]:
import os, shutil, subprocess, sys
from datetime import datetime
from pathlib import Path
from IPython.display import Audio, display

from bg_text_normalizer import normalize as contextual_normalize
from bulgarian_normalization import normalize_for_mfa, normalize_with_punctuation, prosody_words
from num2wordBg import text_numbers_to_words


def normalize_text_for_inference(text):
    if not USE_CUSTOM_TEXT_NORMALIZER or TEXT_NORMALIZER_MODE == 'none':
        return text
    if TEXT_NORMALIZER_MODE == 'legacy':
        return normalize_with_punctuation(text)
    if TEXT_NORMALIZER_MODE == 'contextual':
        return normalize_with_punctuation(contextual_normalize(text))
    raise ValueError("TEXT_NORMALIZER_MODE must be 'contextual', 'legacy', or 'none'")


def preview_text_normalization(text):
    normalized = normalize_text_for_inference(text)
    if SHOW_NORMALIZATION_PREVIEW:
        print('Input text:')
        print(' ', text)
        print('Contextual normalizer output:')
        print(' ', contextual_normalize(text))
        print('Legacy number-expanded only via num2wordBg:')
        print(' ', text_numbers_to_words(text))
        print('Normalized text that tools/infer_bulgarian.py will synthesize:')
        print(' ', normalized)
        print('MFA word-only text after selected normalization:')
        print(' ', normalize_for_mfa(normalized))
        punctuation_tokens = [token for _, token in prosody_words(normalized) if token]
        print('Punctuation/control tokens after selected normalization:')
        print(' ', punctuation_tokens or '(none)')
    return normalized

ckpt_dir = Path(DRIVE_DIR) / 'output_prosody_v2' / 'ckpt' / 'Bulgarian'
result_dir = Path(DRIVE_DIR) / 'inference_results'
preview_text_normalization(TEXT)
output_id = OUTPUT_ID or f"infer_{VOCODER_MODE}_{datetime.now().strftime('%Y%m%d_%H%M%S_%f')}"

cmd = [
    sys.executable, 'tools/infer_bulgarian.py',
    '--text', TEXT,
    '--output-id', output_id,
    '--ckpt-dir', str(ckpt_dir),
    '--result-dir', str(result_dir),
    '--duration-control', str(DURATION_CONTROL),
    '--pitch-control', str(PITCH_CONTROL),
    '--energy-control', str(ENERGY_CONTROL),
    '--vocoder-mode', VOCODER_MODE,
    '--text-normalizer', TEXT_NORMALIZER_MODE if USE_CUSTOM_TEXT_NORMALIZER else 'none',
    '--mfa-cmd', MFA_CMD,
]
if RESTORE_STEP is not None:
    cmd += ['--restore-step', str(RESTORE_STEP)]
if VOCODER_MODE.lower().strip() != 'default':
    cmd += ['--finetuned-vocoder', ACTIVE_VOCODER_PATH]
if INSTALL_MFA_G2P:
    cmd += ['--mfa-bin', MFA_BIN, '--mfa-root-dir', MFA_ROOT_DIR, '--mamba-root-prefix', MAMBA_ROOT]

env = os.environ.copy()
if INSTALL_MFA_G2P:
    env['PATH'] = MFA_BIN + os.pathsep + env['PATH']
    env['MFA_ROOT_DIR'] = MFA_ROOT_DIR
    env['MAMBA_ROOT_PREFIX'] = MAMBA_ROOT

result = subprocess.run(cmd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, env=env)
print(result.stdout)
result.check_returncode()

wav = max(result_dir.glob('*.wav'), key=lambda p: p.stat().st_mtime)
png = wav.with_suffix('.png')
print('VOCODER_MODE:', VOCODER_MODE)
print('output id:', output_id)
print('latest wav:', wav)
if png.is_file():
    print('latest plot:', png)

local_g2p_cache = Path('preprocessed_data/Bulgarian/g2p_cache.json')
drive_g2p_cache = Path(DRIVE_DIR) / 'g2p_cache_prosody_v2.json'
if local_g2p_cache.is_file():
    shutil.copy2(local_g2p_cache, drive_g2p_cache)
    print('saved G2P cache:', drive_g2p_cache)
display(Audio(str(wav), rate=22050))

To synthesize another sentence, change `TEXT` and rerun only the last cell. If you changed the checkpoint, also update `RESTORE_STEP`.